BUONE PRATICHE NELLA SCELTA DEI MODELLI (CNN vs RNN vs ANN)

La scelta non parte dal nome del modello, ma dalla struttura dei dati.
Dati tabellari, senza ordine interno -> MLP
Dati con struttura spaziale -> CNN
Dati ordinati in sequenza -> RNN, LSTM, GRU

*** QUANDO SCEGLIERE UNA MLP ***

MLP (Multi-Layer-Perceptron)
E' la rete neurale composta principalmente da layer Dense
L'input è normalmente una riga di dati trasformata in vettore:
[quantita, prezzo, giorni_consegna, numero_righe, storico_ritardi]
La MLP tratta questi valori come caratteristiche dello stesso record. Non considera automaticamente: la posizione, l'ordine temporale, la vicinanza tra pixel, la sequenza delle parole.
Esempio
vuoi prevedere se un ordine sarà in ritardo
Una MLP può essere appropriata perchè l'ordine è rappresentato da una singola riga di dati

Consiglio:
se ho una taebella ERP non partire sempre e solo da MLP, confronta anche con: - regressione logistica o lineare - Random Forest - XCBoost, LightGBM, CatBoot
Il Deep Learning non è automaticamente superiore (rispetto al machine learning) sui dati tabellari

*** QUANDO SCEGLIERE UNA CNN ***

CNN (Convolutional Neural Network)
La CNN è adatta quando ha importanza la posizione relativa dei dati. Il caso classico è un'immagine.
Un'immagine contiene pixel disposti nello spazio
pixel vicini -> bordo
pixel combinati -> forma
forme combinate -> oggetto
La CNN utilizza filtri che scorrono sull'immagine e identificano pattern locali.
Esempio
fotografica di un raccordo
output 
confrome/non conforme

Prima di addestrare una CNN da zero, valuta il Transfer Learning
ResNet / EfficientNet / MobilNet

Una CNN potrebbe servire per scansioni, fotografie di documenti, pdf convertiti in immagini.
Ma leggere il significato linguistico di una frase non è il compito naturale di una CNN

*** QUANDO SCEGLIERE UNA RNN ***

RNN (Recurrent Neural Network)
E' progettata per dati in cui l'ordine è importante
Esempi: parole in una frase, valori in una serie temporale, misurazioni di un sensore, sequenza di eventi, caratteri di una stringa.
La RNN legge un elemento alla volta mantenendo uno stato interno 
dato attuale + memoria precedente -> nuova memoria
Esempio 1
input: 10,12,13,15,16
output: previsione valore successivo
Esempio 2
frase: 'Il prodotto non è diffettoso'
il significato di 'difettoso' dipende dalla parola 'non' letta prima. Il modello deve conservare la relazione tra le parole.

Una RNN classica oggi è soprattutto didattica
Per un'applicazione reale valuta:
* LSTM
* GRU
* Transformer
* modelli non neurali per le serie temporali

*** QUANDO SCEGLIERE UNA LSTM ***

LSTM()
La LSTM è una particolare RNN con meccanismi di memoria più evoluti.
Utilizza dei date: - forgot get (decide cosa dimenticare) - input gate (decide cosa memorizzare) - output gate (decide cosa usare per il risultato)
E' stata costruita per ridurre il problema della memoria corta delle RNN classiche
Esempio 1 (semplice)
consumi mensili: gennaio, febbraio, marzo, ...
La LSTM tenta di utilizzare la sequenza passata per prevedere i mesi successivi
Esempio 2 (più sensato)
input giornaliero di: temperatura, vibrazione, pressione, velocità macchina, assorbimento elettrico
output: probabilità di guasto nelle prossime 24 ore

Non usare una LSTM solo perchè hai delle date
Questa ipotesi non regge: i dati sono mensili -> uso una LSTM (no)
Se hai 4 anni di dati mensili (4x12=48 osservazioni) non hai un dataset adatto a una LSTM
In questo caso potresti:
- media mobile
- exponential smoothing
- ARIMA o SARIMA
- Croston per domanda intermiettente
- Gradient Boosting con lag
- regole MRP


*** REGOLA GENERALE ***

Tabella → non presumere MLP
Immagine → CNN
Sequenza → RNN/LSTM, ma confrontare alternative
Testo moderno → Transformer
Pochi dati → Transfer Learning
Pochissimi dati → modello semplice o regole


Analisi dei parametri
Efficienza computazionale
Valutare il numero di parametri addestrabili è un passaggio critico
Una rete densa su un'immagine 224 x 224 porterebbe a un'esplosione di connesioni non gestibili
Sarebbe come collegare con un cavo ogni casa di una città con ogni altra casa
La convoluzione risolve questo problema tramite il 'parameter sharing' (usiamo lo stesso filtro su tutta l'immagine), permettendo di analizzare immagini grandi con un numero limitato di pesi.
Come avere un unico stampo per migliaia di mattoni.

Una volta scelto il modello dobbiamo chiederci: come facciamo ad essere sicuri che il risultato non sia frutto del caso?

Il Dogma della Riproducibilità
Determinismo in un mondo stocastico
Nel Deep Learning, l'inizzializzazione dei pesi e il mescolamento dei dati sono processi casuali. Senza un controllo rigoroso, due esecuzioni dello stesso codice produrranno risultati differenti e non comparabili.
La riproducibilità è fondamentale per il dubugging, per la pubblicazione scientifiche e per garantire che un miglioramento della 'accuracy' sia dovuto al modello e non alla fortuna.

Ma dove si nascondono questi dati che dobbiamo bloccare?

Gestione dei Semi Aleatori
I 3 livelli di seed
- Python Seed: impostare 'random.seed()' per tutte le operazioni native del linguaggio e la gestione delle liste
- Numpy Seed: fondamentale per la generazione di array e operazioni matematiche preliminari tramite np.random.seed()
- Backend Seed: il livello più critico (tensorflow o pytorch). Usare 'tf.random.set_seed()' o 'torch.manual_seed()' per l'inizializzazione dei layer e dropout
- Determinismo GPU: alcune operazioni su CUDA non sono deterministiche per ottimizzare la velocità; esistono flag per forzare il determinimo totale (anche a costo di un piccolo calo di performance).

Ma perchè dovremmo rallentare il nostro lavoro per questo?
per il dubugging

Validazione e Verifica
- Debugging: se un modello fallisce o diverge, poter ricreare esattamente l'errore è l'unico modo per identificare se il problema risiede negli iperparametri o nei dati. La riproducibilità è la nostra macchina del tempo.
- Confronto modelli: per dire che l'architettura A è meglio della B, entrambe devono partire dalla stessa identica inizializzazione dei pesi.
- Rigore scientifico: in ambiente di ricerca, i risultati non sono considerati validi se non sono replicabili da terzi partendo dallo stesso seme.

Vediamo come implementare tutto questo con una semplice funzione

Implementazione Universale
Funzione di reset
Una buona pratica è definire una funzione 'set_seeds' all'inizio di ogni notebook o script, garantendo che ogni libreria sia sincronizzata

Come diventare più efficienti
parliamo di risorse

Gestione Risorse e Costi
Efficienza scalalbile
Addestrare modelli non è un'attività a costo zero. La memoria video (VRAM) è la risorsa più scarsa e costosa nel Deep Learning moderno.
Saturare la memoria pora a errori di 'Out of Memory' (OOM) mentre un utilizzo inefficiente allunga i tempi di training aumentando i costi in cloud.

Ma quali sono le leve che possiamo tirare per risparmiare spazio?

Ottimizzazione della Memoria
Tecniche per modelli grandi
* Batch Size:
    ridurre la dimensione del batch diminuisce linearmente l'occupazione della VRAM, ma può rendere il gradiente più rumoroso
* Mixed Precision:
    Utiilzzare il formato 'float16' invece di 'float32' dimezza l'uso della memoria e accellera il training sulle GPU moderne.
* Gradient Accumulation:
    tecnica per simulare batch grandi eseguendo più passi di 'forward' prima di un singolo 'backward' di aggiornamento. ci permette di simulare batch enormi anche su GPU piccole spezzatando i dati in più passaggi
* Profiling:
    utilizzo di strumenti come il 'TensorBoardProfiler' per identificare colli di bottiglia nel caricamento dei dati.

Ma se il nostro budget è limitato?
dobbiamo guardare oltre il codice

Sostenibilità Economica
- Costi Cloud
    monitorare il tempo di addestramento è vitale. Un modello inefficiente può costare migliaia di euro in più in istanze GPU on-demand
    in cloud il temp è denaro
- Checkpoint Stragegici
    Salvare il modello periodicamente evita di perdere ore di lavoro in caso di interruzioni della connessione o crash del sistema
- Pruring e Quantizzazione
    Rimuovere connessioni inutili o ridurre la precisone dei pesi post-training permette di distribuire il modello su dispositivi con risorse limitate.

Calcolo dell'Occupazione
Memoria delle Attivazioni
La memoria non è occupato solo dai pesi del modello, ma soprattutto dalle attivazioni prodote durante il passaggio dei dati, che crescono con la dimensione del batch
Capire questa distinzione permette di progettare architetture che non eccedono le capacità hardware disponibili.


Vediamo delle funzioni di supporto come best-pratiche

In [ ]:
import os

# 1. SET BACKEND (Deve essere fatto prima di importare Keras)
os.environ["KERAS_BACKEND"] = "torch"
 
import keras 
from keras import layers, models, mixed_precision
import random
import numpy as np
import tensorflow as tf


# 1. IL PILASTRO DELLA RIPRODUCIBILITÀ (SEEDING)
def init_reproducibility(seed=42):
    """
    Imposta il seme aleatorio su tutti i livelli del software stack.
    Teoria: L'inizializzazione dei pesi (es. Glorot) usa distribuzioni 
    probabilistiche. Fissando il seed, la 'casuallità' diventa deterministica.
    """
    # Seme per le operazioni native di Python (es. shuffle di liste)
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    
    # Seme per Numpy (gestione dei vettori e preprocessing)
    np.random.seed(seed)
    
    # Seme globale di TensorFlow (inizializzazione pesi e dropout)
    # Nel 2026, questa funzione sincronizza anche i seed dei generatori interni
    tf.keras.utils.set_random_seed(seed)
    
    # Forza TensorFlow a usare operazioni deterministiche su GPU (se disponibili)
    # Nota: Questo può rallentare leggermente il training ma garantisce l'identità dei bit
    tf.config.experimental.enable_op_determinism()
    print(f"[*] Ambiente impostato con Seed: {seed}")

# 2. GESTIONE MEMORIA E COSTI (MIXED PRECISION)
def setup_optimization():
    """
    Abilita la Mixed Precision per dimezzare l'uso della VRAM.
    Teoria: Usa float16 per i calcoli e float32 per le variabili critiche.
    Riduce i costi cloud accelerando il training sulle GPU moderne.
    """
    policy = mixed_precision.Policy('mixed_float16')
    mixed_precision.set_global_policy(policy)
    print(f"[*] Ottimizzazione VRAM: {policy.compute_dtype} abilitata")

# 3. SCELTA DELL'ARCHITETTURA (CNN vs ANN)
def build_model(input_shape, num_classes, architecture_type="CNN"):
    """
    Dimostra la scelta del modello basata sul problema.
    CNN: Analisi spaziale (immagini).
    ANN: Analisi tabulare (dati piatti).
    """
    model = models.Sequential()
    model.add(layers.Input(shape=input_shape))

    if architecture_type == "CNN":
        # Scelta corretta per immagini: sfrutta l'invarianza per traslazione
        # Teoria: I filtri estraggono feature locali indipendentemente dalla posizione
        model.add(layers.Conv2D(32, (3, 3), activation='relu'))
        model.add(layers.MaxPooling2D((2, 2)))
        model.add(layers.Flatten())
    else:
        # ANN Densa: Adatta per dati dove l'ordine dei pixel non è strutturato
        # Teoria: Ogni neurone è connesso a ogni input, ignorando la topologia 2D
        model.add(layers.Flatten())
        model.add(layers.Dense(128, activation='relu'))

    model.add(layers.Dense(num_classes, activation='softmax', dtype='float32'))
    return model

# --- ESECUZIONE PIPELINE ---
init_reproducibility(42)
setup_optimization()

# Caricamento dati (MNIST come esempio di benchmark)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train[..., np.newaxis] / 255.0, x_test[..., np.newaxis] / 255.0

# Costruzione del modello (Scelta: CNN per dati di tipo immagine)
model = build_model((28, 28, 1), 10, architecture_type="CNN")

model.compile(optimizer='adam', 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

# Definiamo un percorso con estensione .keras (Standard 2026)
checkpoint_path = "models/best_mnist_model.keras"

# Creiamo la cartella se non esiste
os.makedirs("models", exist_ok=True)

cp_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_best_only=True,      # Salva solo se l'accuratezza migliora
    monitor='val_accuracy',   # Monitora la precisione di validazione
    mode='max',
    verbose=1
)

# EarlyStopping: Best practice per evitare l'overfitting e risparmiare costi energetici
es_callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

# Esegui il fit con i nuovi callback (aggiungendo validation_split)
model.fit(
    x_train, y_train, 
    epochs=10, 
    batch_size=64, 
    validation_split=0.1,  # Necessario per monitorare val_accuracy
    callbacks=[cp_callback, es_callback]
)

# val_accuracy: 0.9847 - val_loss: 0.0554

[*] Ambiente impostato con Seed: 42
[*] Ottimizzazione VRAM: float16 abilitata
Epoch 1/10
844/844 ━━━━━━━━━━━━━━━━━━━━ 0s 343ms/step - accuracy: 0.8510 - loss: 0.5404
Epoch 1: val_accuracy improved from None to 0.96533, saving model to models/best_mnist_model.keras

Epoch 1: finished saving model to models/best_mnist_model.keras
844/844 ━━━━━━━━━━━━━━━━━━━━ 311s 368ms/step - accuracy: 0.9127 - loss: 0.3099 - val_accuracy: 0.9653 - val_loss: 0.1306
Epoch 2/10
338/844 ━━━━━━━━━━━━━━━━━━━━ 2:37 312ms/step - accuracy: 0.9613 - loss: 0.1466